In [1]:
# ROGII Wellbore Geology — starter: GR-calibrated Viterbi decoder + LightGBM residual
# Flow:  recalibrate typewell GR (per well)  ->  Viterbi decode eval-zone TVT (primary)
#        ->  LightGBM predicts the residual (TVT - decoder)  ->  final = decoder + residual
import os, glob, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from numba import njit
from scipy.signal import savgol_filter
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error
import lightgbm as lgb
warnings.filterwarnings("ignore")

# --- Auto-detect the competition folder (holds train/, test/, sample_submission.csv) ---
def find_data_root():
    cands = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    for c in cands:
        root = Path(c).parent
        if (root / "train").exists() and (root / "test").exists():
            return root
    # fallback to the path the original public notebook used
    return Path("/kaggle/input/rogii-wellbore-geology-prediction")

DATA = find_data_root()
print("Data root:", DATA)
print("train wells:", len(glob.glob(str(DATA/'train'/'*__horizontal_well.csv'))),
      "| test wells:", len(glob.glob(str(DATA/'test'/'*__horizontal_well.csv'))))

FORMATIONS = ["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]

class CFG:
    grid_step = 0.5      # TVT discretization for the decoder (ft). Smaller = finer/slower.
    grid_halfwin = 100.0 # decode only within last_known ± this many ft (kinematic bound)
    jmax_ft = 6.0        # max |TVT jump| allowed per 1-ft MD step (band half-width, ft)
    gr_smooth = 5        # rolling window for GR used in the emission term
    n_splits = 5
    seed = 42

Data root: /kaggle/input/competitions/rogii-wellbore-geology-prediction
train wells: 773 | test wells: 3


In [2]:
# ---- Robust affine calibration of the typewell GR into THIS well's GR space ----
# Fit on the known zone, where we have both true TVT (TVT_input) and GR.
# We compare well-GR against "typewell GR sampled at the known TVTs", then map the
# whole typewell curve into the well's GR units so emission compares like-with-like.
def robust_affine(x, y, iters=3):
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 20 or np.std(x) < 1e-6:
        return 1.0, (float(np.median(y - x)) if len(x) else 0.0)   # offset-only fallback
    a, b = np.polyfit(x, y, 1)
    for _ in range(iters):                                          # trim outliers, refit
        r = y - (a*x + b)
        s = np.median(np.abs(r - np.median(r))) * 1.4826 + 1e-6
        keep = np.abs(r - np.median(r)) < 3*s
        if keep.sum() < 20: break
        a, b = np.polyfit(x[keep], y[keep], 1)
    return float(a), float(b)

# ---- Apparent-dip model from the known zone:  dTVT/dMD ≈ beta*(dZ/dMD) + c ----
# Gives the physically expected TVT step the decoder should follow between points.
def fit_dip(kmd, kz, ktvt):
    dz, dt, dm = np.diff(kz), np.diff(ktvt), np.diff(kmd)
    m = dm > 0
    if m.sum() >= 10 and np.std((dz/dm)[m]) > 1e-6:
        vz, vt = (dz/dm)[m], (dt/dm)[m]
        A = np.column_stack([vz, np.ones_like(vz)])
        (beta, c), *_ = np.linalg.lstsq(A, vt, rcond=None)
        zsig = max(float(np.std(vt - (beta*vz + c))), 1e-3)
        return float(beta), float(c), zsig
    return -1.0, 0.0, 0.1      # default = flat bed (TVT + Z ≈ const)

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))

In [3]:
# ============================================================================
#  Likelihood-weighted multi-seed particle filter (PRIMARY predictor)
#  State per particle: pos = TVT + Z  (the formation datum; ~flat for flat beds)
#                      rate = d(pos)/dMD  (slow drift velocity, momentum walk)
#  TVT estimate = pos - Z.  GR likelihood pins pos against the typewell.
#  We run the whole PF n_seeds times; each seed -> a full track + a total
#  log-likelihood.  We then blend seed tracks weighted by exp(loglik/scale),
#  so the tracks that best explained the observed GR dominate. This is what
#  resists drift/mode-collapse and is the core of every ~6.4 public notebook.
# ============================================================================
from numba import njit

# ---- tuning knobs (raise the first two for the FINAL run) ----
CFG.PF_PARTICLES = 400     # -> 500 for final
CFG.PF_SEEDS     = 64      # -> 128 for final
CFG.CAL_SEEDS    = 16      # seeds used inside self-calibration (cheap)
CFG.SCALES       = (2.0, 4.0, 8.0, 16.0)   # seed-blend temperatures self-cal chooses from
CFG.CUTS         = (0.5, 0.7)              # known-prefix cut points for self-cal
CFG.CAL_MIN_KNOWN = 60     # need at least this many known rows to self-calibrate
CFG.DEFAULT_SCALE = 4.0    # fallback blend temperature when we can't calibrate
CFG.DEFAULT_TRUST = 0.6    # fallback PF trust when we can't calibrate
CFG.PF_INIT_SPR   = 4.5    # initial particle spread (ft) around last-known datum
CFG.TRAIN_EVAL_SUBSET = 250  # how many train wells to score for validation
CFG.GS_MULT = 1.5   # add to Cell 3

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))

@njit(cache=True, nogil=True)
def _interp1(grid, v, vmin, step):
    x = (v - vmin) / step
    i = int(x)
    if i < 0:  return grid[0]
    n = len(grid) - 1
    if i >= n:  return grid[n]
    t = x - i
    return grid[i]*(1.0 - t) + grid[i+1]*t

@njit(cache=True, nogil=True)
def _pf_multiseed(md, z, gr, gg, vmin, step, gs, ls, ir, N, n_seeds, seed_base, init_spr):
    MOM, VN, PN = 0.998, 0.002, 0.005     # velocity momentum, vel-noise, pos-noise
    RESAMP, RP, RR = 0.5, 0.1, 0.001      # resample threshold + jitter
    n = len(md)
    preds = np.empty((n_seeds, n))
    logliks = np.empty(n_seeds)
    tmax = vmin + len(gg) * step
    for s in range(n_seeds):
        np.random.seed(seed_base + s)
        pos = np.empty(N); rate = np.empty(N); w = np.empty(N)
        for j in range(N):
            pos[j]  = ls + init_spr * np.random.randn()
            rate[j] = ir + 0.01 * np.random.randn()
            w[j]    = 1.0 / N
        ll = 0.0; prev_md = md[0] - 1.0
        for i in range(n):
            dm = md[i] - prev_md
            if dm < 1.0: dm = 1.0
            # --- predict (motion model) ---
            for j in range(N):
                rate[j] = MOM*rate[j] + VN*np.random.randn()
                pos[j] += rate[j]*dm + PN*np.random.randn()
                tv = pos[j] - z[i]                      # clamp implied TVT to a sane range
                if   tv < vmin - 100.0: pos[j] = (vmin - 100.0) + z[i]
                elif tv > tmax + 100.0: pos[j] = (tmax + 100.0) + z[i]
            # --- update (GR likelihood) --- avg = sum of w_prev*lk = p(obs_i) ---
            avg = 0.0
            for j in range(N):
                eg = _interp1(gg, pos[j] - z[i], vmin, step)
                d = (gr[i] - eg) / gs; dd = d*d
                if dd > 600.0: dd = 600.0
                lk = np.exp(-0.5*dd)
                if lk < 1e-300: lk = 1e-300
                w[j] *= lk; avg += w[j]
            if avg < 1e-300: avg = 1e-300
            ll += np.log(avg)
            for j in range(N): w[j] /= avg
            # --- resample if effective sample size collapses ---
            neff = 0.0
            for j in range(N): neff += w[j]*w[j]
            if 1.0/neff < RESAMP*N:
                cum = np.empty(N); acc = 0.0
                for j in range(N): acc += w[j]; cum[j] = acc
                u0 = np.random.uniform(0.0, 1.0/N)
                npos = np.empty(N); nrate = np.empty(N); ci = 0
                for j in range(N):
                    u = u0 + j*(1.0/N)
                    while ci < N-1 and cum[ci] < u: ci += 1
                    npos[j] = pos[ci] + RP*np.random.randn()
                    nrate[j] = rate[ci] + RR*np.random.randn()
                for j in range(N):
                    pos[j] = npos[j]; rate[j] = nrate[j]; w[j] = 1.0/N
            # --- point estimate ---
            est = 0.0
            for j in range(N): est += w[j]*(pos[j] - z[i])
            preds[s, i] = est; prev_md = md[i]
        logliks[s] = ll
    return preds, logliks

# JIT warm-up so the first real well isn't paying compile time
_ = _pf_multiseed(np.linspace(1,50,20), np.zeros(20), np.full(20,50.0),
                  np.linspace(45,55,100), 45.0, 0.1, 20.0, 50.0, 0.0, 64, 4, 0, 4.5)
print("PF compiled.")

PF compiled.


In [4]:
def _grid(tw_tvt, tw_gr, step=0.2):
    g = np.arange(float(tw_tvt.min()), float(tw_tvt.max())+step, step)
    return np.interp(g, tw_tvt, tw_gr).astype(np.float64), float(tw_tvt.min()), step

def pf_init_params(known, tw_tvt, tw_gr):
    """Anchor (ls = last datum), drift (ir), and GR noise (gs) from a known slice."""
    last = known.iloc[-1]
    ls = float(last["TVT_input"]) + float(last["Z"])
    grf = known["GR"].interpolate(limit_direction="both").fillna(float(tw_gr.mean()))
    tw_at_k = np.interp(known["TVT_input"].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(grf.values - tw_at_k), 10.0, 60.0)) * CFG.GS_MULT   # emission noise
    tail = known.tail(30)
    dt = np.diff(tail["TVT_input"].values); dz = np.diff(tail["Z"].values); dm = np.diff(tail["MD"].values)
    m = dm > 0
    ir = float(np.median((dt+dz)[m] / dm[m])) if m.sum() >= 3 else 0.0        # datum drift rate
    return ls, ir, gs

def run_pf(md, z, gr, tw_tvt, tw_gr, ls, ir, gs, n_seeds, scale,
           N=None, seed_base=0):
    """Run the multi-seed PF, return (blended_track, logliks, per_seed_preds)."""
    if N is None: N = CFG.PF_PARTICLES
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    preds, ll = _pf_multiseed(md.astype(np.float64), z.astype(np.float64), gr.astype(np.float64),
                              gg, gmin, gst, gs, ls, ir, N, n_seeds, seed_base, CFG.PF_INIT_SPR)
    lw = ll - ll.max(); wts = np.exp(lw/scale); wts /= wts.sum()
    return (wts[:, None]*preds).sum(0), ll, preds

def blend(preds, ll, scale):
    lw = ll - ll.max(); wts = np.exp(lw/scale); wts /= wts.sum()
    return (wts[:, None]*preds).sum(0)

In [5]:
# Hold out the TAIL of the known zone (we know its TVT there), predict it with the PF,
# and use the in-well error to (a) pick the seed-blend temperature `scale`, and
# (b) estimate how much to TRUST the PF vs. the safe kinematic drift baseline.
# This is exactly the top notebooks' "visible-prefix calibration", done honestly.
def self_calibrate(hw, tw_tvt, tw_gr):
    kn = hw[hw["TVT_input"].notna()]
    nkn = len(kn)
    if nkn < CFG.CAL_MIN_KNOWN:
        return CFG.DEFAULT_SCALE, CFG.DEFAULT_TRUST
    grf_full = hw["GR"].interpolate(limit_direction="both").fillna(float(tw_gr.mean()))
    scale_err = {sc: [] for sc in CFG.SCALES}
    naive_err, best_err = [], []
    for c in CFG.CUTS:
        cut = int(nkn * c)
        if cut < 30 or nkn - cut < 20:
            continue
        pk = kn.iloc[:cut]                       # pseudo-known
        pe = kn.iloc[cut:]                        # pseudo-eval (truth = TVT_input)
        ls, ir, gs = pf_init_params(pk, tw_tvt, tw_gr)
        md = pe["MD"].values; z = pe["Z"].values
        gr = grf_full.loc[pe.index].values
        truth = pe["TVT_input"].values.astype(float)
        _, ll, preds = run_pf(md, z, gr, tw_tvt, tw_gr, ls, ir, gs,
                              CFG.CAL_SEEDS, CFG.SCALES[0], seed_base=1000)
        naive_err.append(rmse(truth, np.full(len(truth), float(pk["TVT_input"].iloc[-1]))))
        errs_this_cut = {}
        for sc in CFG.SCALES:
            e = rmse(truth, blend(preds, ll, sc)); scale_err[sc].append(e); errs_this_cut[sc] = e
        best_err.append(min(errs_this_cut.values()))
    if not naive_err:
        return CFG.DEFAULT_SCALE, CFG.DEFAULT_TRUST
    mean_err = {sc: np.mean(v) for sc, v in scale_err.items() if v}
    best_scale = min(mean_err, key=mean_err.get)
    pf_rmse, naive_rmse = float(np.mean(best_err)), float(np.mean(naive_err))

    # sharp gate: PF even slightly better on held-out -> trust ~1; PF worse -> fall back fast
    margin = (naive_rmse - pf_rmse) / (naive_rmse + 1e-6)
    trust = float(np.clip(0.5 + 4.0*margin, 0.1, 1.0))
    return best_scale, trust

In [6]:
def predict_well(hw_path, tw_path, is_train):
    wid = Path(hw_path).stem.replace("__horizontal_well", "")
    try:
        hw = pd.read_csv(hw_path)
        tw = pd.read_csv(tw_path).dropna(subset=["TVT","GR"]).sort_values("TVT").drop_duplicates("TVT")
    except Exception:
        return None
    if not {"MD","X","Y","Z","GR","TVT_input"}.issubset(hw.columns) or len(tw) < 5:
        return None
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(kn) < 15 or len(ev) == 0:
        return None
    if is_train and ("TVT" not in hw.columns or ev["TVT"].isna().all()):
        return None
    tw_tvt = tw["TVT"].to_numpy(np.float64); tw_gr = tw["GR"].to_numpy(np.float64)

    # 1) per-well calibration on the known-zone tail
    best_scale, trust = self_calibrate(hw, tw_tvt, tw_gr)

    # 2) PF over the real eval zone, initialised from the FULL known zone
    ls, ir, gs = pf_init_params(kn, tw_tvt, tw_gr)
    grf = hw["GR"].interpolate(limit_direction="both").fillna(float(tw_gr.mean()))
    md = ev["MD"].values; z = ev["Z"].values; gr = grf.loc[ev.index].values
    pf_pred, _, _ = run_pf(md, z, gr, tw_tvt, tw_gr, ls, ir, gs, CFG.PF_SEEDS, best_scale)

    # 3) safe kinematic baseline (constant datum-drift) + trust blend
    last_known = float(kn["TVT_input"].iloc[-1]); last_md = float(kn["MD"].iloc[-1])
    

    # NEW: constant last-known is a SAFE fallback; kin diverges so never blend into it
    fallback = np.full(len(md), last_known, np.float32)
    final = trust*pf_pred + (1.0 - trust)*fallback
    kin = (ls + ir*(md - last_md)) - z          # keep only as a diagnostic column

    out = {
        "well": wid, "id": [f"{wid}_{i}" for i in ev.index],
        "pred": final.astype(np.float32), "pf_pred": pf_pred.astype(np.float32),
        "kin": kin.astype(np.float32), "last_known": np.float32(last_known),
        "trust": np.float32(trust), "best_scale": np.float32(best_scale),
    }
    if is_train:
        out["tvt_true"] = ev["TVT"].to_numpy(np.float32)
    return pd.DataFrame(out)

In [7]:
import time
from joblib import Parallel, delayed
def run_split(split, is_train, limit=None, n_jobs=1):
    paths = sorted(glob.glob(str(DATA/split/"*__horizontal_well.csv")))
    if limit: paths = paths[:limit]
    def _one(hp):
        wid = Path(hp).stem.replace("__horizontal_well","")
        tp = Path(hp).parent / f"{wid}__typewell.csv"
        return predict_well(hp, str(tp), is_train) if tp.exists() else None
    rows = Parallel(n_jobs=n_jobs, backend="loky")(delayed(_one)(hp) for hp in paths)
    rows = [r for r in rows if r is not None]
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

import os
IS_SUBMIT = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"

if IS_SUBMIT:
    CFG.PF_SEEDS, CFG.PF_PARTICLES = 128, 500   # full quality for the real submission
    VAL_LIMIT = 0                               # skip train validation entirely
else:
    CFG.PF_SEEDS, CFG.PF_PARTICLES = 32, 300    # fast while iterating
    VAL_LIMIT = 120                             # small honest check

if VAL_LIMIT:
    val = run_split("train", True, limit=VAL_LIMIT, n_jobs=4)
    print("\n--- validation on train eval zones ---")
    print(f"Naive        : {rmse(val.tvt_true, val.last_known):.3f}")
    print(f"PF only      : {rmse(val.tvt_true, val.pf_pred):.3f}")
    print(f"PF + self-cal: {rmse(val.tvt_true, val.pred):.3f}")
    print(f"mean trust   : {val.trust.mean():.2f}")

# --- always run test + write submission ---
test_df = run_split("test", False, n_jobs=4)

In [8]:
# --- SCRATCH: GS_MULT sweep. Run manually while iterating; delete before final commit. ---
_saved_skip = globals().get("SKIP_SELFCAL", False)
SKIP_SELFCAL = True                      # PF-only; skips wasted self-cal PF runs
results = {}
for m in [0.7, 1.0, 1.25, 1.5, 2.0]:
    CFG.GS_MULT = m
    v = run_split("train", True, limit=100, n_jobs=4)
    results[m] = rmse(v.tvt_true, v.pf_pred)
    print(f"GS_MULT={m}: PF-only {results[m]:.3f}")
best = min(results, key=results.get)
print(f"\nBest GS_MULT = {best}  ({results[best]:.3f})")
CFG.GS_MULT = best                       # keep the winner for the rest of the notebook
SKIP_SELFCAL = _saved_skip        

GS_MULT=0.7: PF-only 13.463
GS_MULT=1.0: PF-only 12.040
GS_MULT=1.25: PF-only 12.036
GS_MULT=1.5: PF-only 12.344
GS_MULT=2.0: PF-only 12.317

Best GS_MULT = 1.25  (12.036)


In [9]:
# TVT is physically smooth along MD; a light per-well Savitzky–Golay removes PF jitter.
def smooth_group(v, w=31, p=3):
    n = len(v); wl = min(w, n if n % 2 else n-1)
    if wl % 2 == 0: wl -= 1
    return savgol_filter(v, wl, p) if wl >= p+2 else v

parts = []
for _, gdf in test_df.groupby("well", sort=False):
    gg = gdf.copy(); gg["pred"] = smooth_group(gg["pred"].to_numpy(np.float64))
    parts.append(gg)
test_df = pd.concat(parts).sort_index()
print("pred stats:", test_df["pred"].describe()[["min","mean","max"]].to_dict())

pred stats: {'min': 11600.69635660887, 'mean': 11906.395489301067, 'max': 12241.482414369531}


In [10]:
sample = pd.read_csv(DATA/"sample_submission.csv")
sub = sample[["id"]].merge(
    test_df[["id","pred"]].rename(columns={"pred":"tvt"}), on="id", how="left")
# fallback for any id we couldn't predict — use test wells' last-known, not val (val may not exist)
sub["tvt"] = sub["tvt"].fillna(float(test_df["last_known"].mean()))
sub[["id","tvt"]].to_csv("submission.csv", index=False)
print("wrote submission.csv:", sub.shape, "| missing:", int(sub["tvt"].isna().sum()))
sub.head()

wrote submission.csv: (14151, 2) | missing: 0


,id,tvt
0,000d7d20_1442,11747.515300
1,000d7d20_1443,11747.555950
2,000d7d20_1444,11747.592940
3,000d7d20_1445,11747.626582
4,000d7d20_1446,11747.657185
